# Brain Age Batch Inference - Colab GPU

Runs **SFCN**, **SynthBA**, and **MIDIBrainAge** on IXI scans.

**Before running:** Runtime -> Change runtime type -> **GPU (T4)**

| Model | Paper | Preprocessing |
|-------|-------|---------------|
| SFCN | Peng et al. 2021 | N4 + deepbet + MNI affine + crop [160,192,160] |
| SynthBA | Puglisi et al. 2024 | built-in (SynthSeg + ANTs MNI) |
| MIDI | Wood et al. 2024 | built-in (HD-BET + ANTs MNI) |

## 0. Mount Drive and clone repo

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
import os

REPO = "/content/faceage-to-brainage"

if not os.path.exists(REPO):
    os.system(f"git clone https://github.com/kondratevakate/faceage-to-brainage.git {REPO}")
else:
    os.system(f"git -C {REPO} pull --rebase")

os.chdir(REPO)
print("Working dir:", os.getcwd())

## 1. Install dependencies

In [ ]:
import os, sys
from pathlib import Path

import subprocess, sys
pkgs = [
    'numpy', 'scipy', 'pandas', 'matplotlib', 'tqdm', 'Pillow',
    'nibabel', 'hd-bet', 'scikit-image', 'torch', 'torchvision',
    'SimpleITK', 'nilearn', 'deepbet', 'synthba',
    'antspyx', 'monai', 'pingouin', 'seaborn',
]
for pkg in pkgs:
    r = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', pkg],
                       capture_output=True, text=True)
    status = 'ok' if r.returncode == 0 else f'FAILED: {r.stderr[-200:]}'
    print(f'  {pkg}: {status}')

if not os.path.exists('vendor/SFCN/brain_age'):
    os.system('git clone --depth 1 https://github.com/ha-ha-ha-han/UKBiobank_deep_pretrain vendor/SFCN')

# MIDIBrainAge is a git submodule -- check for actual files, not just the dir
if not os.path.exists('vendor/MIDIBrainAge/run_inference.py'):
    os.system('git clone --depth 1 https://github.com/MIDIconsortium/BrainAge vendor/MIDIBrainAge')

# Patch pre_process.py for MONAI >= 1.0 (AddChannel was removed)
pp = Path('vendor/MIDIBrainAge/pre_process.py')
raw = pp.read_text()
if 'AddChannel' in raw and 'class AddChannel' not in raw:
    raw = raw.replace('    AddChannel,\n', '')
    raw = raw.replace('from monai.transforms import AddChannel\n', '')
    shim = (
        'try:\n'
        '    from monai.transforms import AddChannel\n'
        'except ImportError:\n'
        '    class AddChannel:\n'
        '        def __call__(self, x):\n'
        '            import numpy as np\n'
        '            return np.expand_dims(x, 0)\n'
    )
    raw = raw.replace('from monai.transforms import', shim + 'from monai.transforms import', 1)
    pp.write_text(raw)
    print('MIDI pre_process.py patched for MONAI >= 1.0')

print('Install complete. Restarting runtime...')
import IPython
IPython.Application.instance().kernel.do_shutdown(restart=True)


## 2. Verify SFCN weights

The weights ship inside the cloned SFCN repo — no separate download needed.

In [ ]:
import os

SFCN_WEIGHT = "vendor/SFCN/brain_age/run_20190719_00_epoch_best_mae.p"

if os.path.exists(SFCN_WEIGHT):
    print("OK:", SFCN_WEIGHT)
else:
    raise FileNotFoundError(
        f"SFCN weights not found at {SFCN_WEIGHT}. "
        "The file should have been cloned with the SFCN repo in step 1. "
        "Re-run the install cell."
    )

## 2.5. Find your data in Drive

In [ ]:
import os
from pathlib import Path

DRIVE = '/content/drive/MyDrive'

# Show top-level Drive folders
print('=== Drive root ===');
for p in sorted(Path(DRIVE).iterdir()):
    print(' ', p.name, '/' if p.is_dir() else '')

# Search for CSV files that look like manifests
print(chr(10) + '=== CSV files in Drive (first 20) ===')
csvs = list(Path(DRIVE).rglob('*.csv'))[:20]
for p in csvs:
    print(' ', p)

# Search for NIfTI files
print(chr(10) + '=== NIfTI files in Drive (first 5) ===')
niis = list(Path(DRIVE).rglob('*.nii.gz'))[:5]
for p in niis:
    print(' ', p)

## 3. Configure paths

**Edit the variables below** to match your Drive layout.

In [ ]:
import json, os
from pathlib import Path
import pandas as pd

REPO = '/content/faceage-to-brainage'
DRIVE = '/content/drive/MyDrive'

# Structure: brain_mri/t1_only/ixi/manifest.csv + images/
BRAIN_MRI    = f'{DRIVE}/brain_mri/t1_only/ixi'
IXI_MANIFEST = f'{BRAIN_MRI}/manifest.csv'
IXI_IMAGES   = f'{BRAIN_MRI}/images'
RESULTS_DIR  = f'{DRIVE}/brain_mri/brainage_results'
PREPROC_DIR  = '/tmp/sfcn_preproc'

# Auto-find if path is wrong
if not Path(IXI_MANIFEST).exists():
    found = list(Path(DRIVE).rglob('manifest.csv'))
    if found:
        IXI_MANIFEST = str(found[0])
        BRAIN_MRI = str(found[0].parent)
        IXI_IMAGES = f'{BRAIN_MRI}/images'
        print('Auto-found manifest:', IXI_MANIFEST)
    else:
        raise FileNotFoundError('manifest.csv not found in Drive')

df = pd.read_csv(IXI_MANIFEST)
print(f'Manifest OK: {len(df)} rows, columns: {list(df.columns)}')
print(df.head(3).to_string())

os.makedirs(f'{RESULTS_DIR}/ixi', exist_ok=True)
os.makedirs(f'{PREPROC_DIR}/ixi', exist_ok=True)

# Detect image path column and age column from actual manifest
img_col = 't1_path' if 't1_path' in df.columns else 't1_filename'
age_col = 'age' if 'age' in df.columns else 'AGE'
id_col  = 'subject_id' if 'subject_id' in df.columns else 'IXI_ID'
print(f'Using: img_col={img_col}, age_col={age_col}, id_col={id_col}')

config = {
    'runtime': {'device': 'cuda'},
    'sfcn': {
        'model_dir':          f'{REPO}/vendor/SFCN',
        'weight_path':        f'{REPO}/vendor/SFCN/brain_age/run_20190719_00_epoch_best_mae.p',
        'skullstrip_command': 'deepbet',
        'skip_skullstrip':   False,
        'keep_skullstripped': False,
        'n4_correct':         True,
        'register_mni':       True,
        'age_bins':           {'start': 42.0, 'step': 1.0, 'count': 40}
    },
    'datasets': {
        'ixi': {
            'manifest_csv':      IXI_MANIFEST,
            'input_path_column': img_col,
            'chron_age_column':  age_col,
            'subject_id_column': id_col,
            'output_csv':        f'{RESULTS_DIR}/ixi/predictions.csv',
            'preproc_dir':       f'{PREPROC_DIR}/ixi'
        }
    }
}

cfg_path = f'{REPO}/config/local/brain_age_local.json'
os.makedirs(os.path.dirname(cfg_path), exist_ok=True)
Path(cfg_path).write_text(json.dumps(config, indent=2))
print('Config written:', cfg_path)

## 4. Smoke test - 3 scans, all models

In [ ]:
import subprocess, sys

def run_batch(model, tta=False, limit=3, dataset='ixi', resume=False):
    cmd = [
        sys.executable, 'scripts/batch_brainage.py',
        '--model', model,
        '--dataset', dataset,
        '--config', 'config/local/brain_age_local.json',
    ]
    if limit is not None:
        cmd += ['--limit', str(limit)]
    if resume:
        cmd.append('--resume')
    if tta:
        cmd.append('--tta')
    print('>>>', ' '.join(cmd))
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT, text=True)
    try:
        for line in proc.stdout:
            print(line, end='', flush=True)
    except KeyboardInterrupt:
        proc.kill()
        raise
    proc.wait()
    if proc.returncode != 0:
        print(f'FAILED (exit {proc.returncode})')
    return proc.returncode

run_batch('sfcn', limit=3)


In [ ]:
run_batch("synthba", limit=3)

In [ ]:
run_batch("midi", limit=3)

## 5. Check smoke test results

In [ ]:
import pandas as pd
from pathlib import Path

for csv in sorted(Path(RESULTS_DIR, "ixi").glob("*.csv")):
    df = pd.read_csv(csv)
    ok = (df["status"] == "ok").sum()
    print(f"{csv.name}  ({ok}/{len(df)} ok)")
    print(df[["subject_id","chron_age","predicted_age","brain_age_gap","status"]].to_string())
    print()

## 6. Full batch - all IXI scans

Each model writes its own CSV. GPU runtime: ~30 min total for 561 scans.

In [ ]:
run_batch("sfcn", limit=None, resume=True)

In [ ]:
run_batch("synthba", limit=None, resume=True)

In [ ]:
run_batch("midi", limit=None, resume=True)

## 7. Compare models

In [ ]:
import pandas as pd, matplotlib.pyplot as plt
from pathlib import Path

frames = [pd.read_csv(p) for p in Path(RESULTS_DIR, "ixi").glob("*.csv")]
all_df = pd.concat(frames, ignore_index=True)
ok = all_df[all_df["status"] == "ok"].copy()

stats = ok.groupby("model_name").apply(lambda g: pd.Series({
    "n":    len(g),
    "MAE":  g["brain_age_gap"].abs().mean(),
    "bias": g["brain_age_gap"].mean(),
    "r":    g[["chron_age","predicted_age"]].corr().iloc[0,1]
})).round(2)
print(stats)

models = ok["model_name"].unique()
fig, axes = plt.subplots(1, len(models), figsize=(6*len(models), 5))
if len(models) == 1:
    axes = [axes]
for ax, (model, gdf) in zip(axes, ok.groupby("model_name")):
    ax.scatter(gdf["chron_age"], gdf["predicted_age"], alpha=0.4, s=10)
    lims = [gdf["chron_age"].min()-2, gdf["chron_age"].max()+2]
    ax.plot(lims, lims, "k--", lw=0.8)
    mae = gdf["brain_age_gap"].abs().mean()
    ax.set_title(f"{model} / MAE={mae:.1f} y")
    ax.set_xlabel("Chronological age"); ax.set_ylabel("Predicted age")
plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/scatter_all_models.png", dpi=150)
plt.show()